# Notebook 02 — Baseline Models with 5-seed Robustness

Train RF, XGBoost, MLP, BNN across 3 splits with 5 seeds each.

## SetupThis notebook trains 4 model families × 3 splits × 5 seeds = 60 experiments.Total runtime: ~30 minutes on a Colab T4 GPU.

In [ ]:
import sys; sys.path.insert(0, '..')import pandas as pdimport numpy as npimport jsonfrom pathlib import Pathfrom sklearn.metrics import r2_scorefrom src.splits import build_splitsfrom src.models import train_xgb_with_val, make_random_forestdf = pd.read_parquet('../data/chf_clean.parquet')FEATURES = ['Tube Diameter', 'Heated Length', 'Pressure', 'Mass Flux', 'Outlet Quality']X = df[FEATURES].values.astype(np.float32)y = df['CHF'].values.astype(np.float32)

In [ ]:
SEEDS = [42, 123, 456, 789, 1000]results = {'XGB': {}, 'RF': {}}for model_name in ['XGB', 'RF']:    for split_name in ['S1_random', 'S2_holdout_refs', 'S3_high_P']:        results[model_name][split_name] = []for seed in SEEDS:    splits = build_splits(X, y, df, seed)    for split_name, (X_tr, y_tr, X_va, y_va, X_te, y_te) in splits.items():        # XGBoost        m = train_xgb_with_val(X_tr, y_tr, X_va, y_va, seed=seed)        y_pred = m.predict(X_te)        r2 = r2_score(y_te, y_pred)        rmse_pct = float(np.sqrt(np.mean(((y_pred-y_te)/y_te)**2))*100)        results['XGB'][split_name].append({'R2': r2, 'rel_RMSE_pct': rmse_pct})                # Random Forest        m_rf = make_random_forest(seed=seed); m_rf.fit(X_tr, y_tr)        y_pred_rf = m_rf.predict(X_te)        r2_rf = r2_score(y_te, y_pred_rf)        rmse_rf = float(np.sqrt(np.mean(((y_pred_rf-y_te)/y_te)**2))*100)        results['RF'][split_name].append({'R2': r2_rf, 'rel_RMSE_pct': rmse_rf})                print(f'seed={seed} {split_name}: XGB R²={r2:.4f}, RF R²={r2_rf:.4f}')# MLP and BNN: see paper supplementary or full project archive on Zenodo# (PyTorch implementations require GPU)with open('../results/robustness_RF_XGB_reproduced.json', 'w') as f:    json.dump(results, f, indent=2)print('Done.')